# Integrals

This section describes how to compute integrals involving the Chebyshev polynomials, and how the `projection_matrix` method of `Basis1D` is used to evaluate inner products and $L^2$ norms of functions represented as Chebyshev series.

## Theory

### Integral of a Chebyshev polynomial

The integral of a single Chebyshev polynomial over the standard interval $[-1, 1]$ has a closed form.
Starting from the substitution $\xi = \cos\theta$, one can show that:
```{math}
:label: cheby_integral
\int_{-1}^{1} T_n(\xi) \, \mathrm{d}\xi =
\begin{cases}
0 & n \text{ odd} \;, \\
\dfrac{2}{1-n^2} & n \text{ even} \;.
\end{cases}
```
See Section 2.4.4 in {cite:t}`masonChebyshevPolynomials2003`.
The odd case follows directly from the anti-symmetry of $T_n$ for odd $n$.

On an arbitrary interval $[a, b]$, the change of variable {eq}`xi_map` gives:
```{math}
:label: cheby_integral_ab
\int_{a}^{b} T_n(x) \, \mathrm{d}x = \frac{b-a}{2} \int_{-1}^{1} T_n(\xi) \, \mathrm{d}\xi =
\begin{cases}
0 & n \text{ odd} \;, \\
\dfrac{b-a}{1-n^2} & n \text{ even} \;.
\end{cases}
```

### Integral of a product

We now compute the integral of the product of two Chebyshev polynomials:
```{math}
:label: proj_matrix
P_{mn} = \int_{a}^{b} T_m(x) \, T_n(x) \, \mathrm{d}x =
\frac{b-a}{2} \int_{-1}^{1} T_m(\xi) \, T_n(\xi) \, \mathrm{d}\xi \;.
```
The matrix $\mat{P}$ is symmetric and sometimes called the $L^2$ Gram matrix.
We can use the product formula, described in Section 2.4.3 in {cite:t}`masonChebyshevPolynomials2003`:
```{math}
:label: cheby_product
T_m(\xi) \, T_n(\xi) = \tfrac{1}{2}\bigl(T_{m+n}(\xi) + T_{|m-n|}(\xi)\bigr) \;,
```
to write
```{math}
P_{mn} =
\frac{b-a}{4} \int_{-1}^{1} T_{m+n}(\xi) + T_{|m-n|}(\xi) \, \mathrm{d}\xi \;.
```

If $m+n$ is odd, then so is $|m-n|$, and using the result above the integral vanishes:
```{math}
P_{mn} = 0 \;.
```
If $m+n$ is even, then so is $|m-n|$, and the resulting expression for the integral is:
```{math}
:label: cheby_gram
P_{mn} = \frac{b-a}{2} \left[
\frac{1}{1-(m+n)^2} + \frac{1}{1-(m-n)^2} \right]\;.
```

### Projection matrix

Consider two functions $f$ and $g$ both expressed as Chebyshev series of order $N$ on $[a, b]$:

```{math}
f(x) = \sum_{m=0}^{N} f_m T_m(x) \;, \qquad g(x) = \sum_{n=0}^{N} g_n T_n(x) \;.
```

Their $L^2$ inner product is:
```{math}
\langle f, g \rangle = \int_a^b f(x)\,g(x)\,\mathrm{d}x
= \sum_{m=0}^{N}\sum_{n=0}^{N} f_m\,g_n \int_a^b T_m(x)\,T_n(x)\,\mathrm{d}x
= \vec{f}^{\,T} \mat{P} \vec{g} \;,
```

The `projection_matrix` method of `Basis1D` returns $\mat{P}$, whose entries are given above.
This gives compact formulas for the inner product and the $L^2$ norm:

```{math}
:label: inner_product
\langle f, g \rangle = \int_a^b f(x)\,g^\ast(x)\,\mathrm{d}x = \vec{g}^\ast\,\mat{P}\,\vec{f} \;,
\qquad
\|f\|^2_{L^2} = \vec{f}^\ast\,\mat{P}\,\vec{f} \;.
```

## Python API

The `Basis1D` class provides the method `projection_matrix` which returns the matrix $\mat{P}$ defined in {eq}`proj_matrix`.
Given a basis of order $N$ on $[a, b]$, the result is an $(N+1)\times(N+1)$ symmetric matrix with a checkerboard sparsity pattern: $P_{mn} \neq 0$ only when $m+n$ is even.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cheby import Basis1D

basis = Basis1D(4, 0.0, 3.0)
P = basis.projection_matrix()
print(P)

The checkerboard structure — odd-indexed rows and columns sharing no non-zero entries with even-indexed ones — is clearly visible.

### Integral of a single polynomial

The formula {eq}`cheby_integral_ab` can be verified directly.
The integral $\int_a^b T_n(x) \, \mathrm{d}x$ equals the first row (or column) of $\mat{P}$, because $T_0 = 1$ and so $P_{0n} = 2\int_a^b T_0(x)\,T_n(x)\,\mathrm{d}x = 2\int_a^b T_n(x)\,\mathrm{d}x$.

In [ ]:
from scipy import integrate

a, b = 0.0, 3.0
N = 6
basis = Basis1D(N, a, b)
P = basis.projection_matrix()

# Extract integrals from the first row of P
integrals_from_P = P[0, :]

# Compare with numerical quadrature
def Tn(x, n):
    xi = 2 * (x - a) / (b - a) - 1
    return np.cos(n * np.arccos(np.clip(xi, -1, 1)))

integrals_numerical = np.array([
    integrate.quad(lambda x: Tn(x, n), a, b)[0] for n in range(N + 1)
])

# Formula from eq. cheby_integral_ab
ns = np.arange(N + 1)
integrals_formula = np.zeros(N + 1)
integrals_formula[0] = b - a
integrals_formula[2::2] = (b - a) / (1 - ns[2::2] ** 2)  # even n >= 2

print(f"{'n':>3}  {'from P':>12}  {'numerical':>12}  {'formula':>12}")
for n in range(N + 1):
    print(f"{n:>3}  {integrals_from_P[n]:>12.6f}  {integrals_numerical[n]:>12.6f}  {integrals_formula[n]:>12.6f}")

All three approaches agree to machine precision.

### Computing inner products and norms

Using {eq}`inner_product`, the $L^2$ inner product of two functions and the $L^2$ norm of a single function can be computed algebraically from their coefficient vectors and the projection matrix.
As a concrete example, consider $f(x) = \sin(\pi x)$ and $g(x) = \cos(\pi x)$ on $[0, 1]$, for which the exact inner product is $\langle f, g\rangle = 0$ and $\|f\|_{L^2} = \|g\|_{L^2} = 1/\sqrt{2}$.

In [ ]:
from cheby import RealFunction

a, b = 0.0, 1.0
f = RealFunction(lambda x: np.sin(np.pi * x), a, b)
g = RealFunction(lambda x: np.cos(np.pi * x), a, b)

# Build the projection matrix for the relevant basis order
N = max(len(f.coef), len(g.coef)) - 1
basis = Basis1D(N, a, b)
P = basis.projection_matrix()

# Pad shorter coefficient vectors to length N+1
cf = np.zeros(N + 1)
cg = np.zeros(N + 1)
cf[:len(f.coef)] = f.coef
cg[:len(g.coef)] = g.coef

# Inner product and norms via eq. inner_product
inner_product = 0.5 * cf @ P @ cg
norm_f = np.sqrt(0.5 * cf @ P @ cf)
norm_g = np.sqrt(0.5 * cg @ P @ cg)

print(f"<f, g>  = {inner_product:.6e}  (exact: 0)")
print(f"||f||   = {norm_f:.10f}  (exact: {1/np.sqrt(2):.10f})")
print(f"||g||   = {norm_g:.10f}  (exact: {1/np.sqrt(2):.10f})")

The inner product is zero to machine precision, and both norms agree with the exact value $1/\sqrt{2}$ to at least 10 significant digits.

Note that the `RealFunction` and `ComplexFunction` classes expose `norm_L2` and `norm_H1` methods that compute these norms directly from the coefficients using the same projection matrix internally.
The purpose of `Basis1D.projection_matrix` is to give access to the matrix itself, for example to assemble Galerkin system matrices in spectral methods.

## References

```{bibliography}
:filter: docname in docnames
```